# Part 5: Working with large datasets

So far, all the arrays we've worked with fit comfortably in memory. What happens when they don't?

<br><br><br>

## The problem

A 10 000 × 10 000 array of float64 takes about 800 MB.

In [ ]:
import numpy as np

In [ ]:
big = np.random.normal(size=(10_000, 10_000))
print(f"{big.nbytes / 1e9:.1f} GB")

Real scientific and industrial datasets are often hundreds of gigabytes or terabytes. We need strategies for:

1. **Chunking**: processing the array in pieces rather than all at once.
2. **Compression**: storing and reading data efficiently.
3. **Labeled dimensions**: keeping track of what each axis means.

<br><br><br>

## Dask: chunked, parallel arrays

[Dask](https://www.dask.org/) provides a `dask.array` that looks and feels like NumPy, but operates on chunks behind the scenes.

In [ ]:
import dask.array as da

In [ ]:
x = da.random.normal(size=(10_000, 10_000), chunks=(2_500, 2_500))
x

The array hasn't been computed yet. Dask builds a task graph and only computes when you ask for the result.

In [ ]:
result = (x ** 2).mean(axis=1)
result

<br><br><br>

Calling `.compute()` triggers the actual computation.

In [ ]:
result.compute()

<br><br><br>

The key idea: you write the same array-oriented code you already know, and Dask figures out how to split the work across chunks (and cores).

In [ ]:
%%timeit -n1 -r3

np.mean(big ** 2, axis=1)

In [ ]:
x_from_numpy = da.from_array(big, chunks=(2_500, 2_500))

In [ ]:
%%timeit -n1 -r3

da.mean(x_from_numpy ** 2, axis=1).compute()

<br><br><br>

For an array this size, the overhead of Dask's scheduler may make it slower than NumPy. The real advantage shows up when the data is too large to fit in memory, or when you want to use multiple cores or a cluster.

<br><br><br>

## Zarr: chunked, compressed storage

[Zarr](https://zarr.dev/) is an array storage format designed for large datasets. Each chunk is stored as a separate compressed blob, so you can read just the parts you need.

In [ ]:
import zarr

In [ ]:
z = zarr.open("/tmp/example.zarr", mode="w", shape=(10_000, 10_000), chunks=(2_500, 2_500), dtype="f8")
z[:] = big
z.info

<br><br><br>

Zarr arrays support NumPy-style indexing, but read from disk (and decompress) on demand.

In [ ]:
z[0:5, 0:5]

<br><br><br>

Zarr and Dask work well together. You can open a Zarr dataset as a Dask array and process it in parallel without loading everything into memory.

In [ ]:
z_read = zarr.open("/tmp/example.zarr", mode="r")
x_from_zarr = da.from_zarr("/tmp/example.zarr")
x_from_zarr

In [ ]:
da.mean(x_from_zarr ** 2, axis=1).compute()

<br><br><br>

## Blosc2: fast compression and lazy expressions

[Blosc2](https://www.blosc.org/python-blosc2/python-blosc2.html) is a high-performance compression library designed for numerical data. It's often used as a compression codec inside Zarr, but it also has its own array container.

In [ ]:
import blosc2

In [ ]:
big_int = np.random.randint(0, 100, size=(10_000, 10_000), dtype=np.int32)
b = blosc2.asarray(big_int)
print(f"Original:   {big_int.nbytes / 1e6:.0f} MB")
print(f"Compressed: {b.schunk.cbytes / 1e6:.0f} MB")
print(f"Ratio:      {big_int.nbytes / b.schunk.cbytes:.1f}x")

<br><br><br>

Blosc2 is fast enough that reading compressed data and decompressing it can be faster than reading uncompressed data, because less data needs to travel through the memory bus.

In [ ]:
%%timeit -n3 -r3

big_int.sum()

In [ ]:
%%timeit -n3 -r3

b.sum()

<br><br><br>

Blosc2 also supports lazy expressions. You can build an expression tree on compressed arrays and evaluate it without fully decompressing the data first.

In [ ]:
expr = b ** 2 + b
expr

In [ ]:
expr.compute()

<br><br><br>

## xarray: labeled dimensions

[xarray](https://xarray.dev/) adds names and coordinates to array dimensions. Instead of remembering that `axis=0` is time and `axis=1` is latitude, you can refer to them by name.

In [ ]:
import xarray as xr

In [ ]:
temperature = xr.DataArray(
    np.random.normal(20, 5, size=(365, 50, 100)),
    dims=["time", "latitude", "longitude"],
    coords={
        "time": np.arange(np.datetime64("2025-01-01"), np.datetime64("2025-01-01") + 365),
        "latitude": np.linspace(-90, 90, 50),
        "longitude": np.linspace(-180, 180, 100),
    },
)
temperature

<br><br><br>

Select by label instead of integer index:

In [ ]:
temperature.sel(time="2025-07-15", method="nearest")

<br><br><br>

Reduce by dimension name instead of axis number:

In [ ]:
temperature.mean(dim="time")

<br><br><br>

And xarray has a built-in `.plot()` method that automatically labels the axes.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
temperature.mean(dim="time").plot()

None

<br><br><br>

## How they all fit together

These tools compose:

* **Blosc2** compresses chunks of numerical data very quickly.
* **Zarr** organizes compressed chunks into an array-on-disk format.
* **Dask** reads and processes those chunks in parallel, without loading the full array.
* **xarray** wraps the result with labeled dimensions and coordinates.

A typical workflow:

```
disk (Zarr + Blosc2)  →  chunks in parallel (Dask)  →  labeled result (xarray)
```

<br><br><br>

In [ ]:
temperature.chunk({"time": 73, "latitude": 25, "longitude": 50})

Now `temperature` is backed by Dask arrays. All xarray operations on it will be lazy and parallelized.

In [ ]:
chunked = temperature.chunk({"time": 73, "latitude": 25, "longitude": 50})
monthly = chunked.resample(time="ME").mean()
monthly

In [ ]:
monthly.compute()

<br><br><br>

The main takeaway: you can scale array-oriented programming to datasets much larger than your RAM without changing the way you think. The same slicing, broadcasting, and reductions you learned with NumPy work across all of these tools.

The array-oriented paradigm also extends to GPU programming:

* [CuPy](https://cupy.dev/) — a NumPy-compatible array library for NVIDIA GPUs.
* [RAPIDS](https://rapids.ai/) — a suite of GPU-accelerated libraries for data science, including [cuDF](https://docs.rapids.ai/api/cudf/stable/) (DataFrames) and [cuML](https://docs.rapids.ai/api/cuml/stable/) (machine learning).
* [cuPyNumeric](https://docs.nvidia.com/cupynumeric/latest/) — a drop-in NumPy replacement that runs on NVIDIA GPUs and scales to multi-GPU/multi-node.
* [Numba CUDA](https://nvidia.github.io/numba-cuda/) — write GPU kernels in Python using the same Numba we saw in Part 3.
* [JAX](https://jax.readthedocs.io/) — which we already saw in Part 3 — runs the same array-oriented code on CPUs, GPUs, and TPUs.
* [PyTorch](https://pytorch.org/) — its tensor API is array-oriented, and the same code runs transparently on CPU or GPU.
* [CUDA Python](https://github.com/nvidia/cuda-python) — idiomatic Python access to CUDA Runtime and bindings for lower-level GPU programming.
* [NVIDIA CCCL](https://nvidia.github.io/cccl/unstable/python/index.html) — CUDA C++ Core Libraries with Python bindings for GPU-accelerated parallel algorithms.

Array-oriented thinking is the common thread: whether you're scaling _up_ (GPU) or scaling _out_ (Dask), the programming model stays the same.